# InterpretML Implementation
This notebook demonstrates InterpretML on the Medical Insurance dataset.

## Requirements
Run the following to install required dependencies. See `requirements-xai-frameworks.txt` for details.

In [ ]:
!pip install interpret pandas numpy scikit-learn matplotlib==3.5.0

## Setup and Data
Loading the Medical Insurance dataset (dummy tabular data).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from interpret.glassbox import ExplainableBoostingRegressor

# Load dataset (Replace with your path if different)
try:
    df = pd.read_csv('../insurance.csv')
except FileNotFoundError:
    # Dummy data generator if csv not found
    np.random.seed(42)
    df = pd.DataFrame({
        'age': np.random.randint(18, 65, 1000),
        'sex': np.random.choice(['male', 'female'], 1000),
        'bmi': np.random.uniform(18, 40, 1000),
        'children': np.random.randint(0, 5, 1000),
        'smoker': np.random.choice(['yes', 'no'], 1000),
        'region': np.random.choice(['southwest', 'southeast', 'northwest', 'northeast'], 1000),
        'charges': np.random.uniform(1000, 50000, 1000)
    })

categorical_features = ['sex', 'smoker', 'region']
numeric_features = ['age', 'bmi', 'children']

X = df[categorical_features + numeric_features]
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data loaded successfully. Shape:", X.shape)


## Model Training
InterpretML shines with Glassbox models. We'll train an Explainable Boosting Machine (EBM).

In [ ]:
ebm = ExplainableBoostingRegressor(random_state=42)
ebm.fit(X_train, y_train)
print("EBM trained!")


## Explanability Framework API
Following the XAI Platform internal API conventions (like `lime_service.py`), we implement: `create_explainer`, `compute_global_interpret`, and `explain_instance`.

In [ ]:
from interpret import show

class InterpretMLServiceStub:
    @staticmethod
    def create_explainer(model, X_train, y_train):
        # EBM is a glassbox model, but if it were blackbox, we would setup Morris Sensitivity here.
        # For EBM, the model IS the explainer.
        return model
        
    @staticmethod
    def compute_global_interpret(explainer, X_bg):
        global_explanation = explainer.explain_global(name='EBM Global')
        return global_explanation
        
    @staticmethod
    def explain_instance(explainer, instance, y_actual=None):
        local_explanation = explainer.explain_local(instance, y_actual, name='EBM Local')
        return local_explanation

service = InterpretMLServiceStub()
explainer = service.create_explainer(ebm, X_train, y_train)


## Global Explanation

In [ ]:
global_exp = service.compute_global_interpret(explainer, X_train)
# In a real notebook env, 'show' renders an interactive widget
# show(global_exp)
print("Global Feature Importance (Approximate):")
for feature, importance in zip(global_exp.data()['names'], global_exp.data()['scores']):
    print(f"{feature}: {importance:.4f}")


## Local Explanation

In [ ]:
sample_instance = X_test.iloc[0:1]
sample_y = y_test.iloc[0:1]
local_exp = service.explain_instance(explainer, sample_instance, sample_y)
# show(local_exp)

print("Local Explanation for instance 0:")
for i, feature in enumerate(local_exp.data(0)['names']):
    print(f"{feature}: {local_exp.data(0)['scores'][i]:.4f}")
